#### Generate Historical NBA 2K Attributes

##### Imports

In [4]:
import pandas as pd
from pathlib import Path

import joblib

##### Directories and File Locations

In [ ]:
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
GENERATED_DIR = BASE_DIR / "data" / "generated"
MODELS_DIR = BASE_DIR / "models"

HISTORICAL_STATS_PATH = PROCESSED_DIR / "all_historical_nba_stats.json"
PROCESSED_ATTRIBUTES_PATH = PROCESSED_DIR / "all_attributes.csv"
OUTPUT_PATH = GENERATED_DIR / "historical_generated_attributes.csv"

##### Load Historical Stats

In [6]:
historical_df = pd.read_json(HISTORICAL_STATS_PATH)

historical_df = historical_df.rename(
    columns={
        "PLAYER_NAME": "name"
    }
)

historical_df.head()

,name,TEAM_ABBREVIATION,GP_base,W_base,L_base,W_PCT_base,MIN_base,FGM_base,FGA_base,FG_PCT_base,...,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_advanced,FGA_advanced,FGM_PG,FGA_PG,season
0,AJ Price,IND,50,22,28,0.440,15.9,2.3,6.4,0.356,...,95.56,79.63,95.56,0.075,1579,114,320,2.3,6.4,2011
1,Aaron Brooks,PHX,59,26,33,0.441,21.8,3.7,9.9,0.375,...,96.01,80.01,96.01,0.078,2571,220,587,3.7,9.9,2011
2,Aaron Gray,NOH,41,21,20,0.512,12.9,1.4,2.4,0.566,...,90.56,75.47,90.56,0.065,997,56,99,1.4,2.4,2011
3,Acie Law,GSW,51,20,31,0.392,14.2,1.6,3.6,0.435,...,96.68,80.56,96.68,0.070,1460,81,186,1.6,3.6,2011
4,Al Harrington,DEN,73,45,28,0.616,22.8,3.8,9.2,0.416,...,98.16,81.80,98.16,0.078,3389,281,675,3.8,9.2,2011


##### Features

In [7]:
feature_cols = [
    "GP_base",
    "MIN_base",
    "FG_PCT_base",
    "FG3M",
    "FG3A",
    "FG3_PCT",
    "FTM",
    "FTA",
    "FT_PCT",
    "OREB",
    "DREB",
    "REB",
    "AST",
    "TOV",
    "STL",
    "BLK",
    "BLKA",
    "PF",
    "PFD",
    "PTS",
    "PLUS_MINUS",
    "OFF_RATING",
    "DEF_RATING",
    "NET_RATING",
    "AST_PCT",
    "AST_TO",
    "AST_RATIO",
    "OREB_PCT",
    "DREB_PCT",
    "REB_PCT",
    "E_TOV_PCT",
    "EFG_PCT",
    "TS_PCT",
    "USG_PCT",
    "PACE",
    "PIE",
    "POSS",
    "FGM_PG",
    "FGA_PG"
]

##### Labels

In [8]:
target_cols = [
    "overallAttribute",
    "closeShot",
    "midRangeShot",
    "threePointShot",
    "freeThrow",
    "shotIQ",
    "offensiveConsistency",
    "layup",
    "standingDunk",
    "drivingDunk",
    "postHook",
    "postFade",
    "postControl",
    "drawFoul",
    "hands",
    "interiorDefense",
    "perimeterDefense",
    "steal",
    "block",
    "helpDefenseIQ",
    "passPerception",
    "defensiveConsistency",
    "speed",
    "strength",
    "vertical",
    "stamina",
    "hustle",
    "overallDurability",
    "passAccuracy",
    "ballHandle",
    "speedWithBall",
    "passIQ",
    "passVision",
    "offensiveRebound",
    "defensiveRebound",
    "agility"
]

##### Validate Inputs

In [9]:
id_cols = ["name", "TEAM_ABBREVIATION", "season"]

missing_feature_cols = [
    col for col in feature_cols
    if col not in historical_df.columns
]

missing_id_cols = [
    col for col in id_cols
    if col not in historical_df.columns
]

missing_model_files = [
    MODELS_DIR / f"{target_col}_model.pkl"
    for target_col in target_cols
    if not (MODELS_DIR / f"{target_col}_model.pkl").exists()
]

if missing_feature_cols:
    raise ValueError(f"Missing feature columns: {missing_feature_cols}")

if missing_id_cols:
    raise ValueError(f"Missing identifier columns: {missing_id_cols}")

if missing_model_files:
    raise FileNotFoundError(f"Missing model files: {missing_model_files}")

##### Generate Attribute Predictions

In [10]:
X_historical = historical_df[feature_cols]

predicted_attributes = pd.DataFrame(index=historical_df.index)

for target_col in target_cols:
    model = joblib.load(MODELS_DIR / f"{target_col}_model.pkl")
    predicted_attributes[target_col] = model.predict(X_historical)

predicted_attributes.head()

,overallAttribute,closeShot,midRangeShot,threePointShot,freeThrow,shotIQ,offensiveConsistency,layup,standingDunk,drivingDunk,...,hustle,overallDurability,passAccuracy,ballHandle,speedWithBall,passIQ,passVision,offensiveRebound,defensiveRebound,agility
0,73.402031,80.896673,74.105030,69.977716,72.477624,79.535324,74.818441,79.291829,26.702771,50.553142,...,84.907968,79.745485,80.194698,85.921282,81.613454,78.254436,68.477184,36.462280,46.020627,79.252014
1,74.904553,82.937327,73.455505,72.881530,85.550328,77.016605,81.131196,83.267939,23.937687,64.601385,...,84.032265,80.294598,83.835314,87.854840,82.118368,85.168423,75.656796,31.744425,40.100842,80.778863
2,72.545853,68.528836,52.466504,28.825663,49.664204,59.194194,41.607368,64.317977,73.258634,71.355023,...,84.309639,83.749233,41.371055,38.467003,33.611951,52.111281,30.077482,89.278736,85.292366,53.262415
3,72.340636,82.699227,75.092533,62.155315,72.933692,71.569738,62.582163,78.411243,35.777384,65.047437,...,82.209653,81.148596,72.977561,78.153676,76.903431,71.434434,64.103895,36.732014,44.835206,78.858901
4,75.932715,76.751132,71.752074,78.330117,73.951872,75.796805,72.572464,76.583046,50.740394,72.575328,...,84.972889,80.786456,64.283341,64.219950,62.471407,62.920661,37.847968,51.067515,70.683266,69.975441


##### Create Final Dataset

In [11]:
final_df = pd.concat(
    [
        historical_df[id_cols],
        predicted_attributes
    ],
    axis=1
)

final_df.head()

,name,TEAM_ABBREVIATION,season,overallAttribute,closeShot,midRangeShot,threePointShot,freeThrow,shotIQ,offensiveConsistency,...,hustle,overallDurability,passAccuracy,ballHandle,speedWithBall,passIQ,passVision,offensiveRebound,defensiveRebound,agility
0,AJ Price,IND,2011,73.402031,80.896673,74.105030,69.977716,72.477624,79.535324,74.818441,...,84.907968,79.745485,80.194698,85.921282,81.613454,78.254436,68.477184,36.462280,46.020627,79.252014
1,Aaron Brooks,PHX,2011,74.904553,82.937327,73.455505,72.881530,85.550328,77.016605,81.131196,...,84.032265,80.294598,83.835314,87.854840,82.118368,85.168423,75.656796,31.744425,40.100842,80.778863
2,Aaron Gray,NOH,2011,72.545853,68.528836,52.466504,28.825663,49.664204,59.194194,41.607368,...,84.309639,83.749233,41.371055,38.467003,33.611951,52.111281,30.077482,89.278736,85.292366,53.262415
3,Acie Law,GSW,2011,72.340636,82.699227,75.092533,62.155315,72.933692,71.569738,62.582163,...,82.209653,81.148596,72.977561,78.153676,76.903431,71.434434,64.103895,36.732014,44.835206,78.858901
4,Al Harrington,DEN,2011,75.932715,76.751132,71.752074,78.330117,73.951872,75.796805,72.572464,...,84.972889,80.786456,64.283341,64.219950,62.471407,62.920661,37.847968,51.067515,70.683266,69.975441


##### Truncate to two decimals

In [12]:
for col in target_cols:
    final_df[col] = final_df[col].round(2)

##### Export

In [ ]:
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

final_df.to_csv(
    OUTPUT_PATH,
    index=False
)

OUTPUT_PATH

PosixPath('/Users/rushitshah/Downloads/nba-player-attributes/data/generated/historical_generated_attributes.csv')